# Notebook 02 — TextRank Baseline
Run extractive summarization on the full DialogSum test set and evaluate.

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.data.loader import load_dialogsum
from src.methods.textrank import TextRankSummarizer
from src.evaluation.metrics import compute_rouge, compute_bertscore, compute_bleu


## Load Data

In [ ]:
dataset = load_dialogsum(cache_dir='../data/raw/dialogsum')
test = dataset['test']
dialogues  = test['dialogue']
references = test['summary']
print(f'Test samples: {len(dialogues)}')


## Run TextRank

In [ ]:
from tqdm.auto import tqdm

summarizer = TextRankSummarizer(top_n=3)
predictions = []
for d in tqdm(dialogues, desc='TextRank'):
    predictions.append(summarizer.summarize(d))


## Qualitative Examples

In [ ]:
for i in [0, 5, 10]:
    print(f'=== Example {i} ===')
    print('Dialogue :', dialogues[i][:200])
    print('Prediction:', predictions[i])
    print('Reference :', references[i])
    print()


## Evaluation

In [ ]:
rouge_scores = compute_rouge(predictions, references)
print('ROUGE scores:', rouge_scores)


In [ ]:
bs_scores = compute_bertscore(predictions, references)
print('BERTScore:', bs_scores)


In [ ]:
bleu_scores = compute_bleu(predictions, references)
print('BLEU:', bleu_scores)


In [ ]:
results = {
    'scores': {**rouge_scores, 'bertscore_f1': bs_scores['f1'], **bleu_scores},
    'n_samples': len(predictions)
}
with open('../results/scores/textrank_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved to results/scores/textrank_results.json')
print(results['scores'])
